In [1]:
import math
import operator
import pathlib
import random
import utils

from collections import defaultdict, Counter
from functools import reduce
from itertools import product, cycle, islice
from ordered_set import OrderedSet
from string import Template

In [2]:
prenom = utils.read_json("../data/new-lexicon/prenom-modifiers.json")
pp = utils.read_json("../data/new-lexicon/pp-modifiers.json")
lexicon = utils.read_csv_dict("../data/new-lexicon/datives-lexicon-new.csv")

do_template = Template("$agent [verb] $recipient $theme .")
po_template = Template("$agent [verb] $theme to $recipient .")

N = 8

In [3]:
def prenomify(key, form):
    modifiers = prenom[key]
    modified = []
    for m in modifiers:
        modified.append(f"{m} {form}")

    return modified


def ppmodify(key, form):
    modifiers = pp[key]
    modified = []
    for m in modifiers:
        modified.append(f"{form} {m}")

    return modified


def possible_definiteness_forms(entry):
    return {
        "singular": [
            (f"the {entry['singular']}", "definite"),
            (f"a {entry['singular']}", "indefinite"),
        ],
        "plural": [
            (f"the {entry['plural']}", "definite"),
            (f"some {entry['plural']}", "indefinite"),
        ],
    }

def possible_definiteness_forms_structured(entry):
    return {
        "singular": {
            'definite': f"the {entry['singular']}",
            'indefinite': f"a {entry['singular']}"
        },
        "plural": {
            'definite': f"the {entry['plural']}",
            'indefinite': f"some {entry['plural']}"
        }
    }


def definiteness_forms(word, number):
    if number == "singular":
        return [(f"a {word}", "indefinite"), (f"the {word}", "definite")]
    else:
        return [(f"some {word}", "indefinite"), (f"the {word}", "definite")]
    

def definiteness_forms_structured(word, number):
    if number == 'singular':
        return {'definite': f'the {word}', 'indefinite': f'a {word}'}
    else:
        return {'definite': f'the {word}', 'indefinite': f'some {word}'}

In [4]:
definiteness = {"definite": OrderedSet(), "indefinite": OrderedSet()}
animacy = {"animate": OrderedSet(), "inanimate": OrderedSet()}
pronominality = {"pronoun": OrderedSet(), "noun": OrderedSet()}
unique_args = OrderedSet()
form2lemma = {}
lemma2forms = defaultdict(list)
form2number = {}
definite2indefinite = {}

for entry in lexicon:
    lemma, singular, plural, anim, defness, pronom = (
        entry["lemma"],
        entry["singular"],
        entry["plural"],
        entry["animacy"],
        entry["definiteness"],
        entry["pronominality"],
    )
    form2number[singular] = "singular"
    form2number[plural] = "plural"
    # add all the cases that are not supposed
    # to be modified and have fixed definiteness
    if entry["definiteness_flexibility"] == "fixed":
        definiteness[defness].add(singular)
        animacy[anim].add(singular)
        pronominality[pronom].add(singular)
        unique_args.add(singular)
        form2lemma[singular] = lemma
        lemma2forms[lemma].append(singular)
        definite2indefinite[singular] = singular
    else:
        defs = possible_definiteness_forms_structured(entry)
        # add the unmodified forms first
        for k, v in defs.items():
            definite2indefinite[v['definite']] = v['indefinite']
            for definess,form in v.items():
                definiteness[definess].add(form)
                animacy[anim].add(form)
                pronominality[pronom].add(form)
                unique_args.add(form)
                form2lemma[form] = lemma
                lemma2forms[lemma].append(form)

        # modify and then add definiteness
        all_modified_singular = []
        all_modified_plural = []

        # basic prenom modification
        all_modified_singular.extend(prenomify(lemma, singular))
        all_modified_plural.extend(
            prenomify(lemma, plural)
        )  # comment out if we only want singular.

        try:
            # basic pp modification (I think singular only for these)
            all_modified_singular.extend(ppmodify(lemma, singular))
        except:
            continue

        # # combine the two
        # for entry in prenomify(lemma, singular):
        #     try:
        #         pped = ppmodify(lemma, entry)
        #         all_modified_singular.extend(pped)
        #     except:
        #         continue

        # add definiteness to all modified
        for entry in all_modified_singular:
            dfs = definiteness_forms_structured(entry, 'singular')
            definite2indefinite[dfs['definite']] = dfs['indefinite']
            for definess, form in dfs.items():
                definiteness[definess].add(form)
                animacy[anim].add(form)
                pronominality[pronom].add(form)
                unique_args.add(form)
                form2lemma[form] = lemma
                lemma2forms[lemma].append(form)
                form2number[form] = 'singular'

        for entry in all_modified_plural:
            # dfs = definiteness_forms(entry, "plural")
            dfs = definiteness_forms_structured(entry, 'plural')
            definite2indefinite[dfs['definite']] = dfs['indefinite']
            for definess, form in dfs.items():
                definiteness[definess].add(form)
                animacy[anim].add(form)
                pronominality[pronom].add(form)
                unique_args.add(form)
                form2lemma[form] = lemma
                lemma2forms[lemma].append(form)
                form2number[form] = 'plural'

lemma2forms = dict(lemma2forms)
form2number = dict(form2number)
definite2indefinite = dict(definite2indefinite)
indefinite2definite = {v:k for k,v in definite2indefinite.items()}

In [5]:
# all_modified_singular

# definiteness_forms('tool box', "singular")
definite2indefinite['the tool box'], definite2indefinite['mommy'], definite2indefinite['the boy']

('a tool box', 'mommy', 'a boy')

In [6]:
'''
# Sampling strategy:

1. Agent Only
Constraint: cannot have them or it in the arguments + no him/her
remove definite + inanimate + pronouns (them-i/it)
remove them-a/him/her from definite + animate + pronouns
sample items

2. 1arg given
Constraint: cannot have them-i/it/them-a/him/her for the new-argument slot
if arg1 is given, remove definite + inanimate + pronouns from arg2 space
if arg2 is given, remove definite + inanimate + pronouns from arg1 space
sample items

3. 2arg given
Sample items
duplicate and swap argument slot in the duplicated sample set


A caveat: since we are filtering and then sampling, we will get different items in the common space across different given conditions.

We can choose to avoid this by first sampling for unconstrained items:
any definiteness/any animacy/any non-pronoun
indefinite/any animacy/pronouns

potentially problematic:
- pron+def+inanimate; pron+def+animate
'''

'\n# Sampling strategy:\n\n1. Agent Only\nConstraint: cannot have them or it in the arguments + no him/her\nremove definite + inanimate + pronouns (them-i/it)\nremove them-a/him/her from definite + animate + pronouns\nsample items\n\n2. 1arg given\nConstraint: cannot have them-i/it/them-a/him/her for the new-argument slot\nif arg1 is given, remove definite + inanimate + pronouns from arg2 space\nif arg2 is given, remove definite + inanimate + pronouns from arg1 space\nsample items\n\n3. 2arg given\nSample items\nduplicate and swap argument slot in the duplicated sample set\n\n\nA caveat: since we are filtering and then sampling, we will get different items in the common space across different given conditions.\n\nWe can choose to avoid this by first sampling for unconstrained items:\nany definiteness/any animacy/any non-pronoun\nindefinite/any animacy/pronouns\n\npotentially problematic:\n- pron+def+inanimate; pron+def+animate\n'

In [7]:
pronominality_features = pronominality.keys()
animacy_features = animacy.keys()
definiteness_features = definiteness.keys()
givenness_features = [
    "agent-only_none",
    "agent-1arg_theme",
    "agent-1arg_recipient",
    "agent-2arg_both",
]

arg_lengths = OrderedSet([len(item.split(" ")) for item in unique_args])

length_bins = sorted(
    OrderedSet([first - second for first, second in product(arg_lengths, arg_lengths)])
)

single_combo = list(
    product(pronominality_features, animacy_features, definiteness_features)
)
argument_combos = list(product(single_combo, single_combo))


all_features = dict()
for feature_set in [pronominality, animacy, definiteness]:
    for k, v in feature_set.items():
        all_features[k] = v


def feature_intersection(features: tuple):
    items = [all_features[f] for f in features]
    intersected = OrderedSet.intersection(*items)

    return intersected


def feature_string(features):
    string = "".join([f[0] for f in features])
    return string

In [8]:
# first sample all the non problematic ones:

non_constrained = []
constrained = []
for ac in argument_combos:
    if ("pronoun", "inanimate", "definite") in ac:
        constrained.append(ac)
    elif ("pronoun", "animate", "definite") in ac:
        constrained.append(ac)
    else:
        non_constrained.append(ac)

len(constrained), len(non_constrained)

(28, 36)

In [9]:
givenness_constraints = ['them-i', 'it', 'them-a', 'him', 'her']

In [10]:
random.seed(1024)

# non constrained ones: these can be shared across different givenness conditions

unique_combos = 0
string2combo = defaultdict(list)

samples = defaultdict(list)
for gf in givenness_features:
    for ac in argument_combos:
        theme_features, recipient_features = ac
        string = f"{feature_string(theme_features)}{feature_string(recipient_features)}"
        string2combo[string] = [theme_features, recipient_features]

        theme_features = feature_intersection(theme_features)
        recipient_features = feature_intersection(recipient_features)

        pairs = product(theme_features, recipient_features)
        initial_sampling_space = defaultdict(list)

        for item1, item2 in pairs:
            if item1 != item2:  # eliminate the obvious
                lemma1, lemma2 = form2lemma[item1], form2lemma[item2]

                constraint_membership = [
                    item in givenness_constraints for item in (item1, item2)
                ]

                if lemma1 != lemma2:
                    if not (
                        (lemma1 == "me" and lemma2 == "us")
                        or (lemma1 == "us" and lemma2 == "me")
                        or (lemma1 == "them-i" and lemma2 == "them-a")
                        or (lemma1 == "them-a" and lemma2 == "them-i")
                    ):
                        length_diff = len(item1.split(" ")) - len(item2.split(" "))
                        if ("pronoun", "inanimate", "definite") in ac or (
                            "pronoun",
                            "animate",
                            "definite",
                        ) in ac:
                            # constrained -- remove all bad pronouns from things that are *not* given
                            if gf == "agent-only_none" and any(constraint_membership):
                                pass
                            elif (
                                gf == "agent-1arg_theme"
                                and constraint_membership[1] == True
                            ):
                                # pass
                                # print(ac, item1, item2)
                                pass
                            elif (
                                gf == "agent-1arg_recipient"
                                and constraint_membership[0] == True
                            ):
                                pass
                            # non constrained
                            else:
                                initial_sampling_space[length_diff].append(
                                    (item1, item2)
                                )
                        else:
                            initial_sampling_space[length_diff].append((item1, item2))

        initial_sampling_space = dict(initial_sampling_space)

        for k, v in initial_sampling_space.items():
            key = str(k)
            sampled = random.sample(v, min(N, len(v)))
            if len(sampled) < N:
                sampled = list(islice(cycle(sampled), N))

            samples[f"{gf}__{string}_{k}"] = sampled

        unique_combos += len(initial_sampling_space)

string2combo = dict(string2combo)

In [12]:
hims = ["Ross", "Joseph", "Ethan", "Peter", "Thomas"]  # sample some names
hers = ["Lily", "Nina", "Eve", "Catherine", "Sally"]  # sample some names
thems_animate = ["those people", "the children", "the birds", "the ducks", "the pigs"]
thems_inanimate = [
    "the pictures",
    "the crayons",
    "the photographs",
    "the candies",
    "the socks",
]
its = [
    "the picture",
    "the milk",
    "the paper",
    "the apple",
    "the coffee",
]  # sample some object names

given_items = {
    "him": hims,
    "her": hers,
    "them-animate": thems_animate,
    "them-inanimate": thems_inanimate,
    "it": its,
}

# given_templates = {
#     1: {
#         "agent-only": Template("Do you see $agent ?"),
#         "agent-1arg": Template("Do you see $agent and $arg1 ?"),
#         "agent-2arg": Template("Do you see $agent and $arg1 and $arg2 ?"),
#     },
#     2: {
#         "agent-only": Template("Look it's $agent !"),
#         "agent-1arg": Template("Look it's $agent and $arg1 !"),
#         "agent-2arg": Template("Look it's $agent and $arg1 and $arg2 !"),
#     },
#     3: {
#         "agent-only": Template("Here's $agent !"),
#         "agent-1arg": Template("Here's $agent with $arg1 !"),
#         "agent-2arg": Template("Here's $agent with $arg1 and $arg2 !"),
#     },
# }

given_templates = {
    "agent-only": {
        1: Template("Do you see $agent ?"),
        2: Template("Look it's $agent !"),
        3: Template("Here's $agent !"),
    },
    "agent-1arg": {
        1: Template("Do you see $agent and $arg1 ?"),
        2: Template("Look it's $agent and $arg1 !"),
        3: Template("Here's $agent with $arg1 !"),
    },
    "agent-2arg": { # may look weird but there are 4266 occurrences in ao-childes that have the pattern "\b(and)\b(.*)\b(and)\b"
        1: Template("Do you see $agent and $arg1 and $arg2 ?"),
        2: Template("Look it's $agent and $arg1 and $arg2 !"),
        3: Template("Here's $agent with $arg1 !"),
    },
}

agents = ["Laura", "Mark", "Sarah", "William", "Alex", "Judy", "Michael", "Jenny"]

# agents = list(islice(cycle(agents), N))

In [13]:
definiteness_forms('bear', 'singular')

[('a bear', 'indefinite'), ('the bear', 'definite')]

In [14]:
def given_builder(features, form):
    if features[0] == "pronoun":
        # select referents if theme is pronominal
        if form in ["him", "her", "them-a", "them-i", "it"]:
            if form == "them-a":
                form_arg = "them-animate"
            elif form == "them-i":
                form_arg = "them-inanimate"
            else:
                form_arg = form
            given_form = random.sample(given_items[form_arg], 1)[0]
        else:
            given_form = form
    else:
        # modify definite to indefinite in the discourse introduction of the argument
        given_form = definite2indefinite[form]
    return given_form

In [15]:
y = random.sample(range(1, N + 1), int(N / 2))
y, [x for x in range(1, N + 1) if x not in y]

([7, 8, 2, 1], [3, 4, 5, 6])

In [16]:
# raw_stimuli_no_given = []
raw_stimuli = []
idx = 1

for h_id, (combo, items) in enumerate(samples.items()):
    givenness_code, feature_combo = combo.split("__")
    random.shuffle(agents)

    # different givenness orders when both are given:
    theme_recipient = random.sample(range(1, N + 1), int(N / 2))
    recipient_theme = [x for x in range(1, N + 1) if x not in theme_recipient]

    for h_item, (agent, item) in enumerate(zip(agents, items)):
        # raw_stimuli.append(item)
        theme, recipient = item

        do_sentence = do_template.substitute(
            agent=agent, theme=item[0], recipient=item[1]
        )
        po_sentence = po_template.substitute(
            agent=agent, theme=item[0], recipient=item[1]
        )

        do_sentence = do_sentence.replace("them-a", "them")
        po_sentence = po_sentence.replace("them-a", "them")
        do_sentence = do_sentence.replace("them-i", "them")
        po_sentence = po_sentence.replace("them-i", "them")

        string, length_diff = feature_combo.split("_")
        length_diff = int(length_diff)
        theme_features, recipient_features = string2combo[string]

        # loop over givenness codes and then modify arguments in given content appropriately -- e.g., indefinite when first introduced.
        # only do this when given item is definite, when it is not, then ignore?

        if givenness_code == "agent-only_none":
            # do nothing
            for num, template in given_templates["agent-only"].items():
                prefix = template.substitute(agent=agent)
                # add to stimuli
                raw_stimuli.append(
                    {
                        "template_id": num,
                        "item": idx,
                        "hypothesis_id": h_id + 1,
                        "hypothesis_item": h_item + 1,
                        "template_type": "agent-only",
                        "template": "agent-only",
                        "combo": combo,
                        "agent": agent,
                        "theme": item[0],
                        "recipient": item[1],
                        "prefix": prefix,
                        "do_sentence": do_sentence,
                        "po_sentence": po_sentence,
                        "theme_pronominality": theme_features[0],
                        "theme_animacy": theme_features[1],
                        "theme_definiteness": theme_features[2],
                        "recipient_pronominality": recipient_features[0],
                        "recipient_animacy": recipient_features[1],
                        "recipient_definiteness": recipient_features[2],
                        "length_diff": length_diff,
                    }
                )

        elif givenness_code == "agent-1arg_theme" and theme_features[2] == "definite":
            # theme given
            # only add if theme is definite (with required modifications)
            for num, template in given_templates['agent-1arg'].items():
                given_theme = given_builder(theme_features, theme)
                prefix = template.substitute(agent=agent, arg1=given_theme)

                # add to stimuli
                raw_stimuli.append(
                    {
                        "template_id": num,
                        "item": idx,
                        "hypothesis_id": h_id + 1,
                        "hypothesis_item": h_item + 1,
                        "template_type": "agent-1arg",
                        "template": "agent-theme",
                        "combo": combo,
                        "agent": agent,
                        "theme": item[0],
                        "recipient": item[1],
                        "prefix": prefix,
                        "do_sentence": do_sentence,
                        "po_sentence": po_sentence,
                        "theme_pronominality": theme_features[0],
                        "theme_animacy": theme_features[1],
                        "theme_definiteness": theme_features[2],
                        "recipient_pronominality": recipient_features[0],
                        "recipient_animacy": recipient_features[1],
                        "recipient_definiteness": recipient_features[2],
                        "length_diff": length_diff,
                    }
                )

        elif (
            givenness_code == "agent-1arg_recipient"
            and recipient_features[2] == "definite"
        ):
            # recipient given
            # only add if recipient is definite (with required modifications)
            for num, template in given_templates['agent-1arg'].items():
                given_recipient = given_builder(recipient_features, recipient)
                prefix = template.substitute(agent=agent, arg1=given_recipient)

                # add to stimuli
                raw_stimuli.append(
                    {
                        "template_id": num,
                        "item": idx,
                        "hypothesis_id": h_id + 1,
                        "hypothesis_item": h_item + 1,
                        "template_type": "agent-1arg",
                        "template": "agent-recipient",
                        "combo": combo,
                        "agent": agent,
                        "theme": item[0],
                        "recipient": item[1],
                        "prefix": prefix,
                        "do_sentence": do_sentence,
                        "po_sentence": po_sentence,
                        "theme_pronominality": theme_features[0],
                        "theme_animacy": theme_features[1],
                        "theme_definiteness": theme_features[2],
                        "recipient_pronominality": recipient_features[0],
                        "recipient_animacy": recipient_features[1],
                        "recipient_definiteness": recipient_features[2],
                        "length_diff": length_diff,
                    }
                )

        elif (
            givenness_code == "agent-2arg"
            and theme_features[2] == "definite"
            and recipient_features == "definite"
        ):
            # both given, but this can only happen if they are definite
            for num, template in given_templates['agent-2arg'].items():
                given_theme = given_builder(theme_features, theme)
                given_recipient = given_builder(recipient_features, recipient)

                if h_item + 1 in theme_recipient:
                    prefix = template.substitute(
                        agent=agent, arg1=given_theme, arg2=given_recipient
                    )
                    template_name = "agent-theme_recipient"
                else:
                    prefix = template.substitute(
                        agent=agent, arg1=given_recipient, arg2=given_theme
                    )
                    template_name = "agent-recipient_theme"

                raw_stimuli.append(
                    {
                        "template_id": num,
                        "item": idx,
                        "hypothesis_id": h_id + 1,
                        "hypothesis_item": h_item + 1,
                        "template_type": "agent-2arg",
                        "template": template_name,
                        "combo": combo,
                        "agent": agent,
                        "theme": item[0],
                        "recipient": item[1],
                        "prefix": prefix,
                        "do_sentence": do_sentence,
                        "po_sentence": po_sentence,
                        "theme_pronominality": theme_features[0],
                        "theme_animacy": theme_features[1],
                        "theme_definiteness": theme_features[2],
                        "recipient_pronominality": recipient_features[0],
                        "recipient_animacy": recipient_features[1],
                        "recipient_definiteness": recipient_features[2],
                        "length_diff": length_diff,
                    }
                )

        else:
            pass

    idx += 1

In [17]:
# definite2indefinite['the big boys']

len(raw_stimuli)/3
# raw_stimuli

5280.0

In [18]:
print(f"Total Unique Feature Combinations: {unique_combos}")
# print(f"Total stimuli (no-givenness): {len(raw_stimuli_no_given)}")
print(f"Total stimuli (givenness): {len(raw_stimuli)}")
print(f"Total unique stimuli: {int(len(raw_stimuli)/3)}")

Total Unique Feature Combinations: 1356
Total stimuli (givenness): 15840
Total unique stimuli: 5280


In [19]:
def write_stimuli(lst, path):
    stimuli = []
    idx = 0
    for entry in lst:
        stimuli.append(
            {
                "idx": idx + 1,
                "template_id": entry["template_id"],
                "item": entry["item"],
                "dative": "do",
                "hypothesis_id": entry["hypothesis_id"],
                "hypothesis_item": entry["hypothesis_item"],
                "template_type": entry["template_type"],
                "template": entry["template"],
                "combo": entry["combo"],
                "stimulus": f"{entry['prefix']}\n<s> {entry['do_sentence']}",
                "agent": entry["agent"],
                "theme": entry["theme"],
                "recipient": entry["recipient"],
                "theme_pronominality": entry["theme_pronominality"],
                "theme_animacy": entry["theme_animacy"],
                "theme_definiteness": entry["theme_definiteness"],
                "recipient_pronominality": entry["recipient_pronominality"],
                "recipient_animacy": entry["recipient_animacy"],
                "recipient_definiteness": entry["recipient_definiteness"],
                "length_diff": entry["length_diff"],
            }
        )

        stimuli.append(
            {
                "idx": idx + 2,
                "template_id": entry["template_id"],
                "item": entry["item"],
                "dative": "pp",
                "hypothesis_id": entry["hypothesis_id"],
                "hypothesis_item": entry["hypothesis_item"],
                "template_type": entry["template_type"],
                "template": entry["template"],
                "combo": entry["combo"],
                "stimulus": f"{entry['prefix']}\n<s> {entry['po_sentence']}",
                "agent": entry["agent"],
                "theme": entry["theme"],
                "recipient": entry["recipient"],
                "theme_pronominality": entry["theme_pronominality"],
                "theme_animacy": entry["theme_animacy"],
                "theme_definiteness": entry["theme_definiteness"],
                "recipient_pronominality": entry["recipient_pronominality"],
                "recipient_animacy": entry["recipient_animacy"],
                "recipient_definiteness": entry["recipient_definiteness"],
                "length_diff": entry["length_diff"],
            }
        )

        idx += 2

    utils.write_jsonl(stimuli, file_path=path)

In [20]:
template_stimuli = defaultdict(list)
for entry in raw_stimuli:
    template_stimuli[entry["template_id"]].append(entry)

pathlib.Path("../data/experiments/final/").mkdir(exist_ok=True, parents=True)

for k, v in template_stimuli.items():
    write_stimuli(v, path=f"../data/experiments/final/givenness_template_{k}.jsonl")